In [1]:
from BCP303.BCP303 import BPC303
from Sourcemeter.sourcemeter import Sourcemeter2401
import time
from datetime import datetime
from tool.tools import post_process


In [ ]:
# move the stage in Z direction
bcp303_z = BPC303(channel_id=1)
position_z = bcp303_z.move_to_origin(start_position=0)
step_size_z = 0.1  # move 0.1 mm
position_z = bcp303_z.bcp303_move_stage(step_size=step_size_z, current_position=position_z,)
bcp303_z.bcp303_stop(ifback=True)

In [ ]:
# move the stage in x direction
bcp303_x = BPC303(channel_id=2)
position_x = bcp303_x.move_to_origin(start_position=0)
step_size_x = 0.1  # move 0.1 mm
position_x = bcp303_x.bcp303_move_stage(step_size=step_size_x, current_position=position_x,)
bcp303_x.bcp303_stop(ifback=False)

In [ ]:
# record the signals
sm2401 = Sourcemeter2401(speed_nplc=0.1)


In [21]:
# sweep operation: move stage in z-direction and record the signals

def sweep_operation(stage_settings=None,chip_name="chip_test",sample_name="beam_test"):
    step_number = stage_settings["step_number"]
    time_interval = stage_settings["time_interval"]
    position_x = stage_settings["position_x"]
    step_size_z = stage_settings["step_size_z"]
    count = 1
    formatted_time = datetime.now().strftime("%Y%m%d%H%M")
    try:
        bcp303_z = BPC303(channel_id=1)
        bcp303_device = bcp303_z.get_device()
        # moving controller x
        bcp303_x = BPC303(channel_id=2, device=bcp303_device)
        # Sourcemeter
        sm2401 = Sourcemeter2401(speed_nplc=0.1)
        # set the initial position of the stage z
        position_z = bcp303_z.move_to_origin(start_position=stage_settings["position_z"]) - step_size_z
        time.sleep(1)
        bcp303_x.move_to_origin()
        time.sleep(1)
        allData = [{"position": [], "voltage": []}]
        for _ in range(step_number):
            position_z = bcp303_z.bcp303_move_stage(
                step_size=step_size_z,
                current_position=position_z,
            )
            allData[0]["position"].append(position_z)
            time.sleep(0.5)
            bcp303_x.move_to_position(start_position=0, step_size=0.5, final_position=position_x)
            time.sleep(0.5)
            step_start = time.perf_counter()
            voltage = sm2401.measure_voltage(duration=time_interval / 2, dt=0.01)[
                "voltage"
            ]
            allData[0]["voltage"].append(voltage)
            # remaining time to wait until the next step
            elapsed = time.perf_counter() - step_start
            remaining = time_interval - elapsed
            if remaining > 0:
                time.sleep(remaining)
            bcp303_x.move_to_origin()
            time.sleep(0.5)
            print(f"process completed: {100 * count / step_number:.1f}%")
            count += 1
        sm2401_settings = sm2401.getSettings()
        stage_settings["position_x"] = position_x
        settings = {"stage": stage_settings, "sourcemeter": sm2401_settings}
        post_process(
            chip_name=chip_name,
            sample_name=sample_name,
            result=allData[0],
            config=settings,
            repeat=1,
            position_z=position_z,
            ifshow=False,
            formatted_time=formatted_time,
        )
    # stop the stage and sourcemeter
    except Exception as e:
        print(f"Exception occurred: {e}")
    finally:
        time.sleep(0.5)
        sm2401.close()
        bcp303_z.move_to_origin()
        bcp303_z.channel.StopPolling()
        bcp303_x.bcp303_stop(ifback=True)
        return allData[0], settings
    

setting = {
        "position_x": 3,
        "step_number": 30,
        "position_z": 10,
        "step_size_z": 0.1,
        "time_interval": 2,  
    }
sweep_operation(
        stage_settings=setting,
        chip_name="V1_R_W_2_Right",
        sample_name="w10_2_sweep",  
    )

APT High Voltage Amplifier initialized
APT High Voltage Amplifier initialized
IDN: KEITHLEY INSTRUMENTS INC.,MODEL 2401,4614233,B02 Jan 20 2021 10:19:49/B01  /W/N
process completed: 3.3%
process completed: 6.7%
process completed: 10.0%
process completed: 13.3%
process completed: 16.7%
process completed: 20.0%
process completed: 23.3%
process completed: 26.7%
process completed: 30.0%
process completed: 33.3%
process completed: 36.7%
process completed: 40.0%
process completed: 43.3%
process completed: 46.7%
process completed: 50.0%
process completed: 53.3%
process completed: 56.7%
process completed: 60.0%
process completed: 63.3%
process completed: 66.7%
process completed: 70.0%
process completed: 73.3%
process completed: 76.7%
process completed: 80.0%
process completed: 83.3%
process completed: 86.7%
process completed: 90.0%
process completed: 93.3%
process completed: 96.7%
process completed: 100.0%
Config saved to: ./result/V1_R_W_2_Right/202606111156_w10_2_sweep\202606111156_V1_R_W_2_

({'position': [10.0003051850948,
   10.0997955259865,
   10.1998962370678,
   10.2999969481491,
   10.4007080294198,
   10.5008087405011,
   10.6009094515824,
   10.6991790520951,
   10.7992797631764,
   10.8987701040681,
   11.0000915555284,
   11.1001922666097,
   11.200292977691,
   11.3010040589618,
   11.399884029664,
   11.4987640003662,
   11.599475081637,
   11.6995757927183,
   11.7996765037996,
   11.9003875850703,
   12.0017090365307,
   12.1005890072329,
   12.2006897183142,
   12.2995696890164,
   12.4002807702872,
   12.500991851558,
   12.5998718222602,
   12.6999725333415,
   12.8006836146123,
   12.9007843256935],
  'voltage': [[0.01483474,
    0.01476671,
    0.0146651,
    0.01474651,
    0.01511698,
    0.01417641,
    0.01461103,
    0.01472064,
    0.01470234,
    0.01474864,
    0.01451872,
    0.0141263,
    0.01466727,
    0.01487998,
    0.01403997,
    0.01424072],
   [0.01451268,
    0.01486215,
    0.01485693,
    0.01518213,
    0.01467821,
    0.01487086,